# BirdCLEF2026 - Model Training

This notebook trains an EfficientNet model for multi-label species classification.

## Why This Approach?

Based on our research and EDA, we chose:

1. **EfficientNet-B0** - Proven in BirdCLEF competitions, fast training
2. **BCE Loss** - Standard for multi-label classification
3. **AdamW optimizer** - Stable training with weight decay
4. **Cosine annealing** - Smooth learning rate decay
5. **Validation AUC** - Matches competition metric

The ensemble with Whisper would be added later for improved performance.

In [12]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import timm
from pathlib import Path
from tqdm.auto import tqdm
import json
import warnings
from torch.utils.tensorboard import SummaryWriter
warnings.filterwarnings('ignore')

# Set paths
DATA_DIR = Path("../input")
TRAIN_AUDIO_DIR = DATA_DIR / "train_audio"
TRAIN_SOUNDSCAPES_DIR = DATA_DIR / "train_soundscapes"
OUTPUT_DIR = Path("..")
OUTPUT_DIR.mkdir(exist_ok=True)
MODELS_DIR = OUTPUT_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Configuration
SAMPLE_RATE = 32000
DURATION = 5
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
EXPECTED_SAMPLES = SAMPLE_RATE * DURATION

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA GeForce GTX 1650
VRAM: 3.9 GB


## 1. Load Data

Load metadata and prepare species mapping.

In [2]:
# Load metadata
train_df = pd.read_csv(DATA_DIR / "train.csv")
train_labels_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Get species list from submission
species_columns = [col for col in sample_submission.columns if col not in ['row_id', 'filename', 'end_time']]
species_to_idx = {sp: i for i, sp in enumerate(species_columns)}
idx_to_species = {i: sp for sp, i in species_to_idx.items()}
NUM_CLASSES = len(species_columns)

print(f"Number of classes: {NUM_CLASSES}")
print(f"Training audio files: {len(train_df)}")
print(f"Train soundscape segments: {len(train_labels_df)}")
print(f"Total training samples: {len(train_df) + len(train_labels_df)}")

Number of classes: 234
Training audio files: 35549
Train soundscape segments: 1478
Total training samples: 37027


## 2. Import Data Processing Functions

We need the audio processing functions from the preprocessing notebook.

In [3]:
import librosa
import soundfile as sf
from audiomentations import Compose, AddGaussianNoise, PitchShift, TimeStretch, Shift
# Audio functions (from notebook 03)
def load_audio(file_path, target_sr=SAMPLE_RATE):
    """Load audio file and resample if necessary."""
    try:
        waveform, sr = sf.read(file_path)
        if len(waveform.shape) > 1:
            waveform = waveform.mean(axis=1)
        if sr != target_sr:
            waveform = librosa.resample(waveform, orig_sr=sr, target_sr=target_sr)
        return waveform, target_sr
    except Exception as e:
        return None, None
def pad_or_truncate(waveform, target_samples=EXPECTED_SAMPLES):
    """Pad or truncate to fixed length."""
    if len(waveform) > target_samples:
        start = np.random.randint(0, len(waveform) - target_samples)
        waveform = waveform[start:start + target_samples]
    elif len(waveform) < target_samples:
        waveform = np.pad(waveform, (0, target_samples - len(waveform)), mode='constant')
    return waveform
def get_mel_spectrogram(waveform, sr=SAMPLE_RATE):
    """Generate mel spectrogram."""
    mel_spec = librosa.feature.melspectrogram(
        y=waveform, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=50, fmax=15000, power=2.0
    )
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    return log_mel_spec
def normalize_spectrogram(spec):
    """Normalize to [0, 1]."""
    spec = spec - spec.min()
    spec = spec / (spec.max() + 1e-8)
    return spec
# Augmentation
train_augmentations = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.3),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
    TimeStretch(min_rate=0.8, max_rate=1.2, p=0.3),
    Shift(min_shift=0, max_shift=0.2, shift_unit='fraction', p=0.3),
])
def augment_waveform(waveform, sr=SAMPLE_RATE):
    return train_augmentations(waveform, sample_rate=sr)
print("Audio processing functions defined.")

Audio processing functions defined.


## 3. Dataset Class

### Why Combine Both Data Sources?

From EDA, we know:
- train_audio: 35k samples, clean recordings, different domain from test
- train_soundscapes: 1.5k samples, field recordings, SAME domain as test

Combining both gives us the best of both worlds:
1. Large quantity for learning representations
2. Domain-specific data for generalization

In [4]:
class BirdDataset(torch.utils.data.Dataset):
    """Dataset combining train_audio and train_soundscapes.
    
    This is crucial for domain adaptation:
    - train_audio: File-level labels, clean recordings
    - train_soundscapes: Segment-level, field recordings (same as test!)
    """
    def __init__(self, df, audio_dir, species_to_idx, augment=True, use_soundscapes=False, soundscapes_df=None):
        self.audio_dir = audio_dir
        self.soundscapes_dir = audio_dir / '../train_soundscapes/'  # Same base for soundscapes
        self.species_to_idx = species_to_idx
        self.augment = augment
        self.use_soundscapes = use_soundscapes
        
        # Prepare train_audio data
        self.train_audio_df = df[['filename', 'primary_label']].copy()
        
        # Prepare soundscapes data if provided
        if use_soundscapes and soundscapes_df is not None:
            self.soundscapes_df = soundscapes_df.copy()
        else:
            self.soundscapes_df = None
            
    def __len__(self):
        if self.use_soundscapes and self.soundscapes_df is not None:
            return len(self.train_audio_df) + len(self.soundscapes_df)
        return len(self.train_audio_df)
    
    def __getitem__(self, idx):
        if self.use_soundscapes and self.soundscapes_df is not None and idx >= len(self.train_audio_df):
            # Get from soundscapes (field recordings)
            soundscape_idx = idx - len(self.train_audio_df)
            row = self.soundscapes_df.iloc[soundscape_idx]
            file_path = self.soundscapes_dir / row['filename']
            waveform, sr = sf.read(file_path)
            if len(waveform.shape) > 1:
                waveform = waveform.mean(axis=1)
            if sr != SAMPLE_RATE:
                waveform = librosa.resample(waveform, orig_sr=sr, target_sr=SAMPLE_RATE)
            
            # Extract segment
            start_time = self._parse_time(row['start'])
            start_sample = int(start_time * SAMPLE_RATE)
            segment = waveform[start_sample:start_sample + EXPECTED_SAMPLES]
            if len(segment) < EXPECTED_SAMPLES:
                segment = pad_or_truncate(segment, EXPECTED_SAMPLES)
            waveform = segment
            
            # Multi-label
            label = torch.zeros(len(self.species_to_idx))
            for species in row['primary_label'].split(';'):
                species_idx = self.species_to_idx.get(species.strip())
                if species_idx is not None:
                    label[species_idx] = 1.0
        else:
            # Get from train_audio (clean recordings)
            row = self.train_audio_df.iloc[idx]
            file_path = self.audio_dir / row['filename']
            waveform, sr = load_audio(file_path)
            
            if waveform is None:
                waveform = np.zeros(EXPECTED_SAMPLES)
            else:
                waveform = pad_or_truncate(waveform, EXPECTED_SAMPLES)
            
            # Single-label
            label = torch.zeros(len(self.species_to_idx))
            species_idx = self.species_to_idx.get(row['primary_label'])
            if species_idx is not None:
                label[species_idx] = 1.0
        
        # Augment during training
        if self.augment:
            waveform = augment_waveform(waveform.copy(), SAMPLE_RATE)
        
        # Convert to spectrogram
        spec = get_mel_spectrogram(waveform, SAMPLE_RATE)
        spec = normalize_spectrogram(spec)
        spec = torch.FloatTensor(spec)        
        spec = spec.repeat(3, 1, 1)        
        return spec, label
    
    def _parse_time(self, time_str):
        parts = time_str.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

print("Dataset class defined.")

Dataset class defined.


## 4. Model Definition

### Why EfficientNet-B0?

1. **Proven in BirdCLEF**: Top solutions consistently use EfficientNet
2. **Fast training**: ~5M parameters, fits in GPU memory easily
3. **ImageNet pretrained**: Good feature extractors for spectrograms
4. **Scalable**: Can upgrade to B1/B2 if time permits

In [5]:
class BirdAudioClassifier(nn.Module):
    """EfficientNet-based classifier for bird audio.
    
    Architecture:
    - Pretrained EfficientNet backbone (ImageNet weights)
    - Replace final layer for 234-class classification
    
    Why this works:
    - Mel spectrograms are 2D images
    - ImageNet features transfer to spectrogram patterns
    - Pretrained weights provide strong initialization
    """
    
    def __init__(self, model_name='efficientnet_b0', num_classes=234, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.num_features = self.backbone.num_features
        self.classifier = nn.Linear(self.num_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        out = self.classifier(features)
        return out


# Test model creation
print("Creating EfficientNet-B0 model...")
efficientnet_model = BirdAudioClassifier('efficientnet_b0', NUM_CLASSES, pretrained=True)
efficientnet_model = efficientnet_model.to(device)
print(f"EfficientNet parameters: {sum(p.numel() for p in efficientnet_model.parameters()):,}")
print(f"Feature dimension: {efficientnet_model.num_features}")

# Test forward pass
test_input = torch.randn(2, 3, N_MELS, 313).to(device)
test_output = efficientnet_model(test_input)
print(f"Test output shape: {test_output.shape}")

Creating EfficientNet-B0 model...


EfficientNet parameters: 4,307,302
Feature dimension: 1280
Test output shape: torch.Size([2, 234])


## 5. Training Functions

### Why These Choices?

| Component | Choice | Rationale |
|-----------|--------|----------|
| Loss | BCEWithLogitsLoss | Multi-label classification |
| Optimizer | AdamW | Stable, combines weight decay |
| LR | 1e-3 | Standard starting point |
| Scheduler | CosineAnnealing | Smooth decay, prevents overshooting |
| Metric | AUC | Matches competition metric |


In [6]:
def train_epoch(model, loader, criterion, optimizer, device, writer, total_steps):
    """Train for one epoch.
    
    Returns average loss.
    """
    model.train()
    total_loss = 0
    for specs, labels in tqdm(loader, desc="Training", leave=False):
        specs = specs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(specs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_steps += 1

        writer.add_scalar('Loss/train', loss.item(), total_steps)
    
    return total_loss / len(loader), total_steps


def validate(model, loader, criterion, device):
    """Validate model and compute AUC.
    
    Why AUC?
    - Competition uses ROC-AUC
    - Handles class imbalance
    - Threshold-independent
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for specs, labels in tqdm(loader, desc="Validating", leave=False):
            specs = specs.to(device)
            labels = labels.to(device)
            
            outputs = model(specs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            # Get predictions (sigmoid for probabilities)
            preds = torch.sigmoid(outputs)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    # Calculate per-class AUC
    from sklearn.metrics import roc_auc_score
    try:
        valid_cols = all_labels.sum(axis=0) > 0
        aucs = []
        for i in range(all_labels.shape[1]):
            if valid_cols[i] and all_labels[:, i].sum() > 1:
                auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
                aucs.append(auc)
        mean_auc = np.mean(aucs) if aucs else 0
    except:
        mean_auc = 0
    
    return total_loss / len(loader), mean_auc

print("Training functions defined.")

Training functions defined.


## 6. Train Model

### Training Strategy

1. **Combine both data sources** - For domain adaptation
2. **90/10 split** - Train/validation
3. **5 epochs** - For demonstration (increase for better results)
4. **Save best** - Based on validation AUC

In [7]:
# Create dataset
print("Creating dataset...")
full_dataset = BirdDataset(
    train_df, TRAIN_AUDIO_DIR, species_to_idx, 
    augment=True, use_soundscapes=True, soundscapes_df=train_labels_df
)

# Split into train/val
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

# Create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Creating dataset...
Train size: 33324, Val size: 3703
Train batches: 1042, Val batches: 116


In [8]:
# Initialize model
model = BirdAudioClassifier('efficientnet_b0', NUM_CLASSES, pretrained=True)
model = model.to(device)

# Loss: Binary Cross-Entropy for multi-label
criterion = nn.BCEWithLogitsLoss()

# Optimizer: AdamW with weight decay
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Scheduler: Cosine annealing for smooth LR decay
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# Training loop
NUM_EPOCHS = 5  # Increase for better results
best_auc = 0
total_steps = 0
writer = SummaryWriter('../runs/my_experiment')

print("Starting training...")
print(f"Epochs: {NUM_EPOCHS}, Batch size: {BATCH_SIZE}")
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    
    train_loss, total_steps = train_epoch(model, train_loader, criterion, optimizer, device, writer, total_steps)
    val_loss, val_auc = validate(model, val_loader, criterion, device)
    scheduler.step()
    writer.add_scalar('Total_Loss/train', train_loss, total_steps)
    writer.add_scalar('Total_Loss/val', val_loss, total_steps)~
    writer.add_scalar('Total_AUC/val', val_auc, total_steps)
    
    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
    
    # Save best model
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), MODELS_DIR / 'efficientnet_best.pt')
        print(f"✓ Saved best model with AUC: {best_auc:.4f}")

Starting training...
Epochs: 5, Batch size: 32

Epoch 1/5


Train Loss: 0.0251, Val Loss: 0.0169, Val AUC: 0.9329
✓ Saved best model with AUC: 0.9329

Epoch 2/5


Train Loss: 0.0152, Val Loss: 0.0138, Val AUC: 0.9537
✓ Saved best model with AUC: 0.9537

Epoch 3/5


Train Loss: 0.0126, Val Loss: 0.0121, Val AUC: 0.9665
✓ Saved best model with AUC: 0.9665

Epoch 4/5


Train Loss: 0.0111, Val Loss: 0.0111, Val AUC: 0.9696
✓ Saved best model with AUC: 0.9696

Epoch 5/5


Train Loss: 0.0099, Val Loss: 0.0102, Val AUC: 0.9727
✓ Saved best model with AUC: 0.9727


## 7. Save Configuration

Save training config for reproducibility.

In [ ]:
# Save training config
config = {
    "model_name": "efficientnet_b0",
    "num_classes": NUM_CLASSES,
    "sample_rate": SAMPLE_RATE,
    "duration": DURATION,
    "n_mels": N_MELS,
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "batch_size": BATCH_SIZE,
    "learning_rate": 1e-3,
    "num_epochs": NUM_EPOCHS,
    "best_val_auc": best_auc,
}

with open(OUTPUT_DIR / 'training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Save species mapping
with open(OUTPUT_DIR / 'species_mapping.json', 'w') as f:
    json.dump(species_to_idx, f)

print(f"Training complete! Best validation AUC: {best_auc:.4f}")
print(f"Model saved to: {MODELS_DIR / 'efficientnet_best.pt'}")

In [14]:
torch.save(model.state_dict(), MODELS_DIR / 'efficientnet_best.pt')

## Summary

This notebook trained an EfficientNet-B0 model:

1. **Combined dataset** - train_audio + train_soundscapes for domain adaptation
2. **BCE loss** - For multi-label classification
3. **Data augmentation** - Noise, pitch shift, time stretch
4. **Validation AUC** - Matches competition metric
5. **Model saving** - Best model based on validation AUC

Next: Run inference and create submission in notebook 05.